**Razonamiento con Agentes de Lenguaje en LangChain**

Este programa permite utilizar el framework [LangChain](https://python.langchain.com/) de OpenAI para desarrollar aplicaciones simples basadas en LLMs. LangChain permite conectar un LLM a otras fuentes de datos (i.e., bases de datos SQL, buscador de Google, etc), y permite  al  LLM interactuar con su ambiente utilizando [Agentes](https://python.langchain.com/docs/modules/agents).

*LangChain* permite generar cadenas de *razonamiento* paso-a-paso para responder a tareas de alto nivel, descomponiéndolas en tareas más simples.
Para lograr esto, un agente tiene acceso a varias herramientas y determina cuál de ellas debe utilizar dependiendo de la entrada del usuario. En general, existen dos tipos de agentes:

1. **Agentes de Acción**: decide la próxima acción utilizando las salidas de las acciones previas. Estos son adecuados para tareas pequeñas.
2. **Agentes de Planificar-y-Ejecutar**: decide sobre la secuencia completa de acciones y luego las ejecuta todas sin actualizar el plan. Estos son adecuados para tareas complejas que requieren mantener objetivos de largo plazo.

Instalamos algunos paquetes tales como LangChain, openai, y buscadores de google:

In [ ]:
!pip install  langchain openai pymysql --upgrade -q
!pip install  google-search-results -q

In [ ]:
!pip install -U langchain-community

Importamos algunas librerías de LangChain para uso de agentes de languaje  y ajustamos las variables de ambiente para uso de las respectivas APIs de OpenAI (**OPENAI_API_KEY**) y Google (**SERPAPI_API_KEY**), para las cuales Ud. debe obtener las respectivas claves:

In [ ]:
from langchain.agents import load_tools
from langchain.agents import initialize_agent
from langchain.agents import AgentType
from langchain.llms import OpenAI
import os

In [ ]:
# Open AI API-key
from google.colab import files
from IPython.display import clear_output

files.upload() # subir archivo con apikey de openai propio
clear_output() # no muestra contenido del apikey

In [ ]:
def get_api_key():
    with open('idsa_openai_key.txt', 'r') as fp: #acá reemplazar x el nombre de tu archivo
        key = fp.read()
    return key

# Enter your OpenAI API key here
OPENAI_API_KEY = get_api_key()

In [ ]:
os.environ['OPENAI_API_KEY']  = OPENAI_API_KEY
os.environ["SERPAPI_API_KEY"] = "2e903e74f87729885b46d4f2d7be6949003031f2a1adbd145da65435cf04b3ab"

Inicializamos las bibliotecas de OpenAI para LLMs y algunas herramientas a utilizar tales como el buscador de Google (*serpAPI*) y un LLM (por defecto un modelo pre-entrenado de GPT como "*text-davinci-003*"):

In [ ]:
llm = OpenAI()
tools = load_tools(["serpapi", "llm-math"], llm=llm)

En este ejemplo,  utilizamos un agente  simple de lenguaje SIN memoria (*ZERO_SHOT_REACT_DESCRIPTION*). Es decir, la acción que este  realiza se basa solamente en la acción actual y no en las previas (historial). De este modo, el agente decide qué herramienta utilizar basado exclusivamente en la descripción de la herramienta:


In [ ]:
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION)
agent.run("¿Quién es la esposa del presidente de Croacia y cuál será la edad de ella en 10 años más?")

'Sanja Musić Milanović tendrá 66 años en 10 años más.'

Ahora, solicitamos al agente que muestre paso-a-paso lo que realizó (verbose):

In [ ]:
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)
agent.run("¿Quién es la esposa del presidente de Croacia y cuál será la edad de ella en 10 años más?")



> Entering new AgentExecutor chain...
 I should use the search engine to find information about the president of Croatia and his wife.
Action: Search
Action Input: "presidente de croacia esposa"
Observation: ['Kolinda Grabar-Kitarović is a Croatian politician and diplomat who served as the president of Croatia from 2015 to 2020. She was the first woman to be ...', 'El 8 de marzo de 2008 se convirtió en embajadora de Croacia en Estados Unidos. ... El 19 de enero de 2015 asumió como la primera mujer presidente de Croacia.', "Kolinda Grabar-Kitarović, službeni Instagram profil. Kolinda Grabar-Kitarović, the official Instagram account. Follow. Message. UNGA2024's profile picture.", 'Kolinda Grabar-Kitarovic, la popular presidenta "hincha" de Croacia a la que acusan de defender políticas xenófobas - BBC News Mundo.', 'UNA MUJER LLAMADA KOLINDA. · Está casada desde 1996 con el empresario Jakob Kitarović, con quien tiene dos hijos · AL FRENTE.', 'Sanja Musić Milanović lleva años en el papel

"The president's wife will be 66 years old in 10 years."